In [1]:
# ================================================================
# 11_CWQ_Dataset_Audit_and_Pilot.ipynb
#
# Runtime: CPU
#
# New CWQ extension only:
# - Downloads/reuses the NSM CWQ archive
# - Audits train/dev/test graph structure and eligibility
# - Runs symbolic baselines on 50 DEV questions
# - Does NOT train a model
# - Does NOT evaluate learned policies on TEST
# - Does NOT change WebQSP data, models, thresholds, or results
# ================================================================

import collections
import hashlib
import json
import random
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
from pathlib import Path

# ----------------------------
# Configuration
# ----------------------------

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")

MAX_HOPS = 4
MAX_FRONTIER = 16
PILOT_DEV_QUESTIONS = 50
SEED = 42

# CWQ.tgz from the NSM authors' public dataset folder.
CWQ_FILE_ID = "1ua7h88kJ6dECih6uumLeOIV9a3QNdP-g"
EXPECTED_ARCHIVE_BYTES = 914072382

REQUIRED_FILES = [
    "train_simple.json",
    "dev_simple.json",
    "test_simple.json",
    "entities.txt",
    "relations.txt",
]

EXPECTED_SPLIT_COUNTS = {
    "train": 27639,
    "dev": 3519,
    "test": 3531,
}

# ----------------------------
# File helpers
# ----------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024**2),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    temporary = path.with_name(path.name + ".partial")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(
            value,
            file,
            indent=2,
            ensure_ascii=False,
        )
    temporary.replace(path)


def iter_jsonl(path):
    with open(path, encoding="utf-8") as file:
        for line_number, text in enumerate(file, 1):
            if not text.strip():
                continue

            try:
                row = json.loads(text)
                if not isinstance(row, dict):
                    raise ValueError("Expected a JSON object")
            except Exception as exc:
                raise ValueError(
                    f"Invalid JSON object: {path}:{line_number}"
                ) from exc

            yield line_number, row


# ----------------------------
# Dataset schema helpers
# ----------------------------

def entity_id(value):
    if isinstance(value, dict):
        value = value.get(
            "kb_id",
            value.get("id", value.get("entity")),
        )

    if isinstance(value, int) and not isinstance(value, bool):
        return value if 0 <= value < len(ENTITIES) else None

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    return None


def relation_id(value):
    if isinstance(value, int) and not isinstance(value, bool):
        return value if 0 <= value < len(RELATIONS) else None

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def get_question_id(split, line_number, row):
    value = row.get("id", row.get("qid"))

    if value is None or not str(value).strip():
        raise ValueError(
            f"Missing question ID in {split}, line {line_number}"
        )

    return str(value)


def parse_example(row):
    question = row.get("question")
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Missing nonempty question")

    subgraph = row["subgraph"]
    if (
        not isinstance(subgraph, dict)
        or not isinstance(subgraph.get("entities"), list)
    ):
        raise ValueError("Missing subgraph entity list")

    nodes = {
        entity_id(value)
        for value in subgraph["entities"]
    }
    if None in nodes:
        raise ValueError("Unmapped graph entity")

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples"),
    )
    if not isinstance(raw_edges, list):
        raise ValueError("Missing subgraph tuples/triples list")

    # Keep the published direction. Do not add inverse edges.
    unique_edges = set()

    for edge in raw_edges:
        if (
            not isinstance(edge, (list, tuple))
            or len(edge) != 3
        ):
            raise ValueError(
                f"Unexpected edge schema: {str(edge)[:100]}"
            )

        source = entity_id(edge[0])
        relation = relation_id(edge[1])
        target = entity_id(edge[2])

        if source is None or relation is None or target is None:
            raise ValueError(
                f"Unmapped edge: {str(edge)[:100]}"
            )

        unique_edges.add((source, relation, target))

    endpoints = {
        node
        for source, relation, target in unique_edges
        for node in (source, target)
    }

    extra_endpoints = len(endpoints - nodes)

    # Edge endpoints are valid graph nodes even if omitted from
    # the separately listed entity array. Record such cases.
    nodes |= endpoints

    raw_answers = row["answers"]
    if not isinstance(raw_answers, list):
        raise ValueError("Answers must be a list")

    mapped_answers = [
        entity_id(value)
        for value in raw_answers
    ]
    gold = {
        value
        for value in mapped_answers
        if value is not None
    }

    raw_topics = row.get(
        "entities",
        row.get("topic_entities", []),
    )
    if not isinstance(raw_topics, list):
        raise ValueError("Topic entities must be a list")

    mapped_topics = [
        entity_id(value)
        for value in raw_topics
    ]
    topics = {
        value
        for value in mapped_topics
        if value is not None
    }

    return {
        "question": question.strip(),
        "nodes": nodes,
        "edges": sorted(unique_edges),
        "gold": gold,
        "topics": topics,
        "unmapped_gold": mapped_answers.count(None),
        "unmapped_topics": mapped_topics.count(None),
        "raw_edges": len(raw_edges),
        "extra_endpoints": extra_endpoints,
    }


# ----------------------------
# Directed graph helpers
# ----------------------------

def build_adjacency(edges):
    grouped = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for source, relation, target in edges:
        grouped[source][relation].add(target)

    return {
        source: {
            relation: sorted(targets)
            for relation, targets in grouped[source].items()
        }
        for source in sorted(grouped)
    }


def shortest_distances(topics, adjacency):
    depth = {node: 0 for node in topics}
    queue = collections.deque(sorted(topics))

    while queue:
        source = queue.popleft()
        source_depth = depth[source]

        if source_depth >= MAX_HOPS:
            continue

        for targets in adjacency.get(source, {}).values():
            for target in targets:
                if target not in depth:
                    depth[target] = source_depth + 1
                    queue.append(target)

    return depth


def text_tokens(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " "),
        )
    )


def relation_overlap(question, relation):
    return len(
        text_tokens(question)
        & text_tokens(RELATIONS[relation])
    )


def get_frontier(depth, expanded, adjacency):
    # An action retrieves all outgoing targets for one
    # (source entity, relation) branch.
    return [
        (source, relation, depth[source], targets)
        for source in sorted(depth)
        if depth[source] < MAX_HOPS
        for relation, targets in sorted(
            adjacency.get(source, {}).items()
        )
        if (source, relation) not in expanded
    ]


def rank_by_relation(branches, question):
    return sorted(
        branches,
        key=lambda branch: (
            -relation_overlap(question, branch[1]),
            branch[2],
            -len(branch[3]),
            branch[1],
            branch[0],
        ),
    )


# ----------------------------
# Diagnostic oracle
# ----------------------------

def future_answer_masks(unseen_gold, adjacency, expanded):
    """
    Bitset reachability through still-unexpanded actions.

    levels[h][node] represents unseen gold entities reachable
    from node within at most h further directed hops.

    Gold is used ONLY for this symbolic diagnostic, not as an
    input to a learned model.
    """
    current = {
        gold_entity: 1 << index
        for index, gold_entity in enumerate(sorted(unseen_gold))
    }

    levels = [current]

    for _ in range(MAX_HOPS - 1):
        following = dict(current)

        for source, by_relation in adjacency.items():
            mask = current.get(source, 0)

            for relation, targets in by_relation.items():
                if (source, relation) in expanded:
                    continue

                for target in targets:
                    mask |= current.get(target, 0)

            if mask:
                following[source] = mask

        current = following
        levels.append(current)

    return levels


def branch_gain(branch, levels):
    source, relation, source_depth, targets = branch
    remaining_hops = MAX_HOPS - (source_depth + 1)

    mask = 0
    for target in targets:
        mask |= levels[remaining_hops].get(target, 0)

    return mask.bit_count()


# ----------------------------
# Symbolic development pilot
# ----------------------------

def symbolic_search(example, policy, action_cap, frontier_cap):
    adjacency = build_adjacency(example["edges"])
    gold = example["gold"]

    depth = {
        topic: 0
        for topic in example["topics"]
    }

    # Oracle answer recognition isolates graph exploration.
    found = gold.intersection(depth)

    expanded = set()
    actions = 0
    retrieved_edges = 0
    frontier_truncations = 0
    stop_reason = "action_cap"

    question_seed = int(
        hashlib.sha256(
            (example["qid"] + policy).encode()
        ).hexdigest()[:8],
        16,
    )
    rng = random.Random(SEED + question_seed)

    while actions < action_cap:
        full_frontier = get_frontier(
            depth,
            expanded,
            adjacency,
        )

        if not full_frontier:
            stop_reason = "empty_frontier"
            break

        candidates = rank_by_relation(
            full_frontier,
            example["question"],
        )

        if frontier_cap is not None:
            frontier_truncations += int(
                len(candidates) > frontier_cap
            )
            candidates = candidates[:frontier_cap]

        if policy == "oracle":
            levels = future_answer_masks(
                gold - found,
                adjacency,
                expanded,
            )

            gains = [
                branch_gain(branch, levels)
                for branch in candidates
            ]

            best_index = max(
                range(len(candidates)),
                key=lambda index: gains[index],
            )

            if gains[best_index] == 0:
                stop_reason = "oracle_zero_reachable_gain"
                break

            chosen = candidates[best_index]

        elif policy == "relation":
            chosen = candidates[0]

        elif policy == "fifo":
            chosen = min(
                full_frontier,
                key=lambda branch: (
                    branch[2],
                    branch[0],
                    branch[1],
                ),
            )

        elif policy == "random":
            chosen = rng.choice(candidates)

        else:
            raise ValueError(f"Unknown policy: {policy}")

        source, relation, source_depth, targets = chosen
        expanded.add((source, relation))

        for target in targets:
            depth[target] = min(
                depth.get(target, MAX_HOPS + 1),
                source_depth + 1,
            )

        found |= gold.intersection(targets)

        retrieved_edges += len(targets)
        actions += 1

    unseen = gold - found

    # Check the FULL remaining frontier, not only the shortlist.
    full_frontier = get_frontier(depth, expanded, adjacency)

    if unseen:
        levels = future_answer_masks(
            unseen,
            adjacency,
            expanded,
        )
        premature = any(
            branch_gain(branch, levels) > 0
            for branch in full_frontier
        )
    else:
        premature = False

    recall = len(found) / len(gold)

    # With oracle recognition, predictions contain only correct
    # visited entities. This is NOT learned end-to-end answer F1.
    oracle_f1 = 2 * recall / (1 + recall)

    return {
        "qid": example["qid"],
        "policy": policy,
        "action_cap": action_cap,
        "frontier_cap": frontier_cap,
        "gold_count": len(gold),
        "found_count": len(found),
        "recall": recall,
        "oracle_recognition_f1": oracle_f1,
        "exact": int(found == gold),
        "actions": actions,
        "retrieved_edges": retrieved_edges,
        "premature": int(premature),
        "frontier_truncations": frontier_truncations,
        "stop_reason": stop_reason,
    }


# ----------------------------
# Main workflow
# ----------------------------

def run_cwq_audit():
    from google.colab import drive
    drive.mount("/content/drive")

    start_time = time.time()

    raw_dir = ROOT / "data/raw/nsm_cwq"
    data_dir = ROOT / "data/extracted/cwq"
    output_dir = ROOT / "outputs/cwq_audit_v1"

    for directory in (raw_dir, data_dir, output_dir):
        directory.mkdir(parents=True, exist_ok=True)

    archive = raw_dir / "CWQ.tgz"

    print("=" * 90)
    print("SETSTOP NOTEBOOK 11: CWQ DATASET AUDIT AND DEV PILOT")
    print("=" * 90)
    print("Runtime: CPU")
    print("No model training or inference.")
    print("WebQSP files, models, and thresholds remain unchanged.")
    print("Maximum graph hops:", MAX_HOPS)
    print("Frontier shortlist:", MAX_FRONTIER)
    print("Development pilot questions:", PILOT_DEV_QUESTIONS)
    print("Data directory:", data_dir)
    print("Output directory:", output_dir)
    print("=" * 90)

    # ----------------------------
    # Download and extract if needed
    # ----------------------------

    extracted_complete = all(
        (data_dir / filename).is_file()
        for filename in REQUIRED_FILES
    )

    if not extracted_complete:
        if not archive.is_file():
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "gdown==5.2.0",
                ],
                check=True,
            )

            import gdown

            partial_archive = archive.with_name(
                archive.name + ".partial"
            )

            print(
                "\nDownloading CWQ.tgz (~0.91 GB). "
                "No manual upload is required."
            )

            result = gdown.download(
                id=CWQ_FILE_ID,
                output=str(partial_archive),
                quiet=False,
            )

            if result is None or not partial_archive.is_file():
                raise RuntimeError(
                    "CWQ download failed. Rerun later. "
                    "No existing models were changed."
                )

            downloaded_bytes = partial_archive.stat().st_size
            if downloaded_bytes != EXPECTED_ARCHIVE_BYTES:
                raise RuntimeError(
                    f"Unexpected archive size: {downloaded_bytes}. "
                    f"Expected {EXPECTED_ARCHIVE_BYTES}. "
                    "The incomplete/different archive was not extracted."
                )

            partial_archive.replace(archive)

        if archive.stat().st_size != EXPECTED_ARCHIVE_BYTES:
            raise RuntimeError(
                "The cached CWQ archive has an unexpected byte count. "
                "It has been preserved; extraction was not attempted."
            )

        print("\nExtracting the five files needed for this audit.")

        # Extract only whitelisted regular files. Avoid extractall.
        with tempfile.TemporaryDirectory(
            prefix="setstop_cwq_extract_"
        ) as temporary_directory:
            staging = Path(temporary_directory)
            seen = set()

            with tarfile.open(archive, "r|gz") as tar:
                for member in tar:
                    filename = Path(member.name).name

                    if filename not in REQUIRED_FILES:
                        continue

                    if not member.isfile() or filename in seen:
                        raise RuntimeError(
                            "Unsafe or ambiguous archive member: "
                            f"{member.name}"
                        )

                    seen.add(filename)
                    source = tar.extractfile(member)

                    if source is None:
                        raise RuntimeError(
                            f"Cannot read archive member: {member.name}"
                        )

                    with source, open(staging / filename, "wb") as file:
                        shutil.copyfileobj(
                            source,
                            file,
                            length=8 * 1024**2,
                        )

            missing = set(REQUIRED_FILES) - seen
            if missing:
                raise RuntimeError(
                    f"Required files missing from archive: {sorted(missing)}"
                )

            for filename in REQUIRED_FILES:
                destination = data_dir / filename
                staged_file = staging / filename

                if destination.exists():
                    if (
                        sha256_file(destination)
                        != sha256_file(staged_file)
                    ):
                        raise RuntimeError(
                            f"Existing file differs: {destination}. "
                            "It was preserved, not overwritten."
                        )
                else:
                    temporary = destination.with_name(
                        destination.name + ".partial"
                    )
                    shutil.copyfile(staged_file, temporary)
                    temporary.replace(destination)

    # ----------------------------
    # Record input identities
    # ----------------------------

    print("\nRecording source-file SHA-256 hashes.")
    print("These are provenance hashes, not a published checksum.")

    source_files = {}
    for filename in REQUIRED_FILES:
        path = data_dir / filename
        print("Hashing:", filename, flush=True)

        source_files[filename] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }

    configuration = {
        "code_version": "cwq_audit_v1_2",
        "seed": SEED,
        "max_hops": MAX_HOPS,
        "frontier_cap": MAX_FRONTIER,
        "pilot_dev_questions": PILOT_DEV_QUESTIONS,
        "answer_range": [2, 50],
        "inverse_edges_added": False,
        "source_files": source_files,
    }

    signature = hashlib.sha256(
        json.dumps(
            configuration,
            sort_keys=True,
        ).encode()
    ).hexdigest()

    # ----------------------------
    # Load global entity/relation IDs
    # ----------------------------

    global ENTITIES, RELATIONS, ENTITY_TO_ID, RELATION_TO_ID

    def read_vocabulary(filename):
        with open(data_dir / filename, encoding="utf-8") as file:
            return [
                line.rstrip("\r\n")
                for line in file
            ]

    ENTITIES = read_vocabulary("entities.txt")
    RELATIONS = read_vocabulary("relations.txt")

    ENTITY_TO_ID = {
        value: index
        for index, value in enumerate(ENTITIES)
    }
    RELATION_TO_ID = {
        value: index
        for index, value in enumerate(RELATIONS)
    }

    if (
        len(ENTITY_TO_ID) != len(ENTITIES)
        or len(RELATION_TO_ID) != len(RELATIONS)
    ):
        raise ValueError(
            "Duplicate vocabulary entries prevent unambiguous ID mapping."
        )

    print("\nEntity vocabulary:", len(ENTITIES))
    print("Relation vocabulary:", len(RELATIONS))

    # ----------------------------
    # Audit each split
    # ----------------------------

    reports = {}
    manifests = {}

    counter_keys = [
        "questions",
        "graph_nodes",
        "input_edges",
        "unique_edges",
        "unmapped_gold_records",
        "extra_edge_endpoints",
        "multi_answer_2_to_50",
        "all_gold_in_graph",
        "main_eligible",
        "eligible_gold",
        "reachable_eligible_gold",
        "fully_reachable_questions",
        "partly_reachable_questions",
        "zero_reachable_questions",
        "eligible_nodes",
        "eligible_input_edges",
    ]

    for split in ("train", "dev", "test"):
        cache_path = output_dir / f"cwq_{split}_audit_cache_v1.json"

        cache = (
            json.loads(cache_path.read_text(encoding="utf-8"))
            if cache_path.exists()
            else None
        )

        if cache and cache.get("signature") == signature:
            reports[split] = cache["report"]
            manifests[split] = cache["manifest"]

            print(
                f"\n{split}: reusing completed audit "
                "(unchanged inputs and configuration)."
            )
            continue

        counts = collections.Counter({
            key: 0 for key in counter_keys
        })
        answer_histogram = collections.Counter()
        manifest = []
        seen_qids = set()

        split_file = data_dir / f"{split}_simple.json"
        print(f"\nAuditing {split}...")

        for line_number, row in iter_jsonl(split_file):
            qid = get_question_id(split, line_number, row)

            if qid in seen_qids:
                raise ValueError(
                    f"Duplicate question ID within {split}: {qid}"
                )
            seen_qids.add(qid)

            try:
                example = parse_example(row)
            except Exception as exc:
                raise ValueError(
                    f"Schema problem: split={split}, "
                    f"line={line_number}, qid={qid}. "
                    f"Details: {exc}"
                ) from exc

            counts["questions"] += 1
            counts["graph_nodes"] += len(example["nodes"])
            counts["input_edges"] += example["raw_edges"]
            counts["unique_edges"] += len(example["edges"])
            counts["unmapped_gold_records"] += example["unmapped_gold"]
            counts["extra_edge_endpoints"] += example["extra_endpoints"]

            gold = example["gold"]
            answer_histogram[str(len(gold))] += 1

            complete_gold_mapping = (
                example["unmapped_gold"] == 0
                and bool(gold)
            )

            multi_answer = (
                complete_gold_mapping
                and 2 <= len(gold) <= 50
            )

            all_gold_in_graph = (
                multi_answer
                and gold <= example["nodes"]
            )

            valid_topics = (
                bool(example["topics"])
                and example["unmapped_topics"] == 0
                and example["topics"] <= example["nodes"]
            )

            eligible = all_gold_in_graph and valid_topics

            counts["multi_answer_2_to_50"] += int(multi_answer)
            counts["all_gold_in_graph"] += int(all_gold_in_graph)

            reachable_gold_count = None
            max_reachable_gold_hop = None

            if eligible:
                adjacency = build_adjacency(example["edges"])
                distances = shortest_distances(
                    example["topics"],
                    adjacency,
                )

                reachable_gold = gold.intersection(distances)
                reachable_gold_count = len(reachable_gold)

                max_reachable_gold_hop = max(
                    (
                        distances[node]
                        for node in reachable_gold
                    ),
                    default=None,
                )

                counts["main_eligible"] += 1
                counts["eligible_gold"] += len(gold)
                counts["reachable_eligible_gold"] += len(reachable_gold)

                counts["fully_reachable_questions"] += int(
                    reachable_gold == gold
                )
                counts["partly_reachable_questions"] += int(
                    bool(reachable_gold)
                    and reachable_gold != gold
                )
                counts["zero_reachable_questions"] += int(
                    not reachable_gold
                )

                counts["eligible_nodes"] += len(example["nodes"])
                counts["eligible_input_edges"] += example["raw_edges"]

            normalized_question = " ".join(
                example["question"].lower().split()
            )

            manifest.append({
                "qid": qid,
                "line_number": line_number,
                "question_sha256": hashlib.sha256(
                    normalized_question.encode()
                ).hexdigest(),
                "gold_count": len(gold),
                "complete_gold_mapping": complete_gold_mapping,
                "eligible": bool(eligible),
                "reachable_gold_count": reachable_gold_count,
                "max_reachable_gold_hop": max_reachable_gold_hop,
            })

            if counts["questions"] % 1000 == 0:
                elapsed = (time.time() - start_time) / 60
                print(
                    f"{split}: {counts['questions']:,} audited | "
                    f"{counts['main_eligible']:,} eligible | "
                    f"{elapsed:.1f} minutes",
                    flush=True,
                )

        total = counts["questions"]
        eligible_total = counts["main_eligible"]

        report = {
            **dict(counts),
            "expected_nsm_questions": EXPECTED_SPLIT_COUNTS[split],
            "expected_question_count_matches": (
                total == EXPECTED_SPLIT_COUNTS[split]
            ),
            "answer_count_histogram": dict(answer_histogram),
            "mean_graph_nodes": (
                counts["graph_nodes"] / max(1, total)
            ),
            "mean_input_edges": (
                counts["input_edges"] / max(1, total)
            ),
            "mean_eligible_input_edges": (
                counts["eligible_input_edges"]
                / max(1, eligible_total)
            ),
            "eligible_fraction": (
                eligible_total / max(1, total)
            ),
            "reachable_gold_fraction": (
                counts["reachable_eligible_gold"]
                / max(1, counts["eligible_gold"])
            ),
            "fully_reachable_fraction": (
                counts["fully_reachable_questions"]
                / max(1, eligible_total)
            ),
        }

        reports[split] = report
        manifests[split] = manifest

        save_json(
            cache_path,
            {
                "signature": signature,
                "report": report,
                "manifest": manifest,
            },
        )

        print(json.dumps({split: report}, indent=2))

        if not report["expected_question_count_matches"]:
            print(
                "WARNING: split size differs from the NSM README. "
                "Review this before training."
            )

    # ----------------------------
    # Check split separation
    # ----------------------------

    split_checks = {}

    for first, second in [
        ("train", "dev"),
        ("train", "test"),
        ("dev", "test"),
    ]:
        first_ids = {
            row["qid"] for row in manifests[first]
        }
        second_ids = {
            row["qid"] for row in manifests[second]
        }

        first_questions = {
            row["question_sha256"]
            for row in manifests[first]
        }
        second_questions = {
            row["question_sha256"]
            for row in manifests[second]
        }

        split_checks[f"{first}_vs_{second}"] = {
            "qid_overlap": len(first_ids & second_ids),
            "normalized_question_overlap": len(
                first_questions & second_questions
            ),
        }

    print("\nSplit checks:")
    print(json.dumps(split_checks, indent=2))

    if any(
        check["qid_overlap"] > 0
        for check in split_checks.values()
    ):
        raise RuntimeError(
            "Cross-split question ID overlap found. "
            "Resolve it before training."
        )

    manifest_path = output_dir / "cwq_manifest_v1.json"

    save_json(
        manifest_path,
        {
            "signature": signature,
            "configuration": configuration,
            "splits": manifests,
            "split_checks": split_checks,
        },
    )

    # ----------------------------
    # Select development pilot
    # ----------------------------

    eligible_dev = [
        row
        for row in manifests["dev"]
        if row["eligible"]
    ]

    # Deterministic selection independent of oracle performance.
    selected = sorted(
        eligible_dev,
        key=lambda row: hashlib.sha256(
            f"{SEED}:{row['qid']}".encode()
        ).hexdigest(),
    )[:PILOT_DEV_QUESTIONS]

    selected_ids = {
        row["qid"] for row in selected
    }

    if not selected_ids:
        raise RuntimeError(
            "No eligible multi-answer development questions. "
            "Inspect the audit; do not start training."
        )

    # Do not remove difficult/partly unreachable selected questions.
    experiments = [
        ("fifo_b12", "fifo", 12, None),
        ("relation_b12", "relation", 12, MAX_FRONTIER),
        ("random_b12", "random", 12, MAX_FRONTIER),
        ("oracle_b12", "oracle", 12, MAX_FRONTIER),
        ("oracle_b24", "oracle", 24, MAX_FRONTIER),
        ("oracle_full_frontier_b24", "oracle", 24, None),
    ]

    print("\nRunning development-only symbolic pilot...")
    print(
        "Gold answers are used for diagnostic recognition/oracle ranking."
    )
    print(
        "These are NOT learned end-to-end model results."
    )

    per_question = []

    for line_number, row in iter_jsonl(
        data_dir / "dev_simple.json"
    ):
        qid = get_question_id("dev", line_number, row)

        if qid not in selected_ids:
            continue

        example = parse_example(row)
        example["qid"] = qid

        for method, policy, action_cap, frontier_cap in experiments:
            result = symbolic_search(
                example,
                policy,
                action_cap,
                frontier_cap,
            )
            result["method"] = method
            per_question.append(result)

        completed = len(per_question) // len(experiments)

        if completed % 10 == 0 or completed == len(selected):
            elapsed = (time.time() - start_time) / 60
            print(
                f"Dev pilot: {completed}/{len(selected)} questions | "
                f"{elapsed:.1f} minutes",
                flush=True,
            )

    expected_rows = len(selected) * len(experiments)
    if len(per_question) != expected_rows:
        raise RuntimeError(
            f"Pilot row mismatch: {len(per_question)} "
            f"instead of {expected_rows}"
        )

    # ----------------------------
    # Aggregate and save
    # ----------------------------

    metrics = [
        "recall",
        "oracle_recognition_f1",
        "exact",
        "actions",
        "retrieved_edges",
        "premature",
        "frontier_truncations",
    ]

    baseline_report = {}

    for method, _, _, _ in experiments:
        rows = [
            row
            for row in per_question
            if row["method"] == method
        ]

        baseline_report[method] = {
            metric: sum(row[metric] for row in rows) / len(rows)
            for metric in metrics
        }

    per_question_path = (
        output_dir / "cwq_dev_pilot_per_question_v1.jsonl"
    )
    temporary = per_question_path.with_name(
        per_question_path.name + ".partial"
    )

    with open(temporary, "w", encoding="utf-8") as file:
        for row in per_question:
            file.write(json.dumps(row) + "\n")
    temporary.replace(per_question_path)

    pilot_path = (
        output_dir / "cwq_dev_pilot_baselines_v1.json"
    )

    save_json(
        pilot_path,
        {
            "signature": signature,
            "split": "dev",
            "questions": len(selected),
            "selected_qids": sorted(selected_ids),
            "oracle_answer_recognition": True,
            "greedy_oracle_not_optimal": True,
            "results": baseline_report,
        },
    )

    summary_path = (
        output_dir / "cwq_dataset_audit_summary_v1.json"
    )

    summary = {
        "signature": signature,
        "source": "NSM CWQ 1.1 release",
        "source_url": (
            "https://github.com/RichardHGL/WSDM2021_NSM"
        ),
        "archive_file_id": CWQ_FILE_ID,
        "data_dir": str(data_dir),
        "configuration": configuration,
        "splits": reports,
        "split_checks": split_checks,
        "dev_pilot": baseline_report,
        "pilot_questions": len(selected),
        "minutes": (time.time() - start_time) / 60,
        "next_step": (
            "Review audit and development oracle headroom "
            "before generating controller training states."
        ),
        "caveats": [
            (
                "Gold is used only for structural auditing "
                "and symbolic oracle diagnostics."
            ),
            (
                "Graph reachability does not enforce CWQ logical "
                "constraints or prove semantic correctness."
            ),
            (
                "The pilot is not learned end-to-end performance; "
                "no test search policies were evaluated."
            ),
            (
                "Gold-based filtering defines a conditional "
                "multi-answer subset, not official full-CWQ accuracy."
            ),
            (
                "Model-text leakage, state-label correctness, and "
                "candidate truncation require later dedicated audits."
            ),
        ],
    }

    save_json(summary_path, summary)

    inventory_path = output_dir / "artifact_inventory_v1.json"

    artifact_paths = [
        summary_path,
        manifest_path,
        pilot_path,
        per_question_path,
        *[
            output_dir / f"cwq_{split}_audit_cache_v1.json"
            for split in ("train", "dev", "test")
        ],
    ]

    save_json(
        inventory_path,
        {
            "signature": signature,
            "files": [
                {
                    "name": path.name,
                    "path": str(path),
                    "bytes": path.stat().st_size,
                    "sha256": sha256_file(path),
                }
                for path in artifact_paths
            ],
        },
    )

    print("\n" + "=" * 105)
    print("CWQ DEV SYMBOLIC PILOT — ORACLE ANSWER RECOGNITION")
    print("=" * 105)

    print(
        f"{'Method':31s} "
        f"{'Recall':>9s} "
        f"{'Oracle F1':>10s} "
        f"{'Exact':>9s} "
        f"{'Actions':>10s} "
        f"{'Edges':>11s}"
    )

    for method, values in baseline_report.items():
        print(
            f"{method:31s} "
            f"{values['recall']:9.4f} "
            f"{values['oracle_recognition_f1']:10.4f} "
            f"{values['exact']:9.4f} "
            f"{values['actions']:10.2f} "
            f"{values['retrieved_edges']:11.2f}"
        )

    print("\nCompleted. WebQSP artifacts remain unchanged.")
    print("No GPU training was performed.")
    print("No learned test-set evaluation was performed.")

    print("\nSEND THESE THREE FILES:")
    print(summary_path)
    print(pilot_path)
    print(inventory_path)

    print(
        "\nDo not start CWQ model training yet. "
        "First review graph coverage and development pilot results."
    )
    print("=" * 105)


run_cwq_audit()

Mounted at /content/drive
SETSTOP NOTEBOOK 11: CWQ DATASET AUDIT AND DEV PILOT
Runtime: CPU
No model training or inference.
WebQSP files, models, and thresholds remain unchanged.
Maximum graph hops: 4
Frontier shortlist: 16
Development pilot questions: 50
Data directory: /content/drive/MyDrive/SetStop_KGQA/data/extracted/cwq
Output directory: /content/drive/MyDrive/SetStop_KGQA/outputs/cwq_audit_v1



Downloading...
From (original): https://drive.google.com/uc?id=1ua7h88kJ6dECih6uumLeOIV9a3QNdP-g
From (redirected): https://drive.google.com/uc?id=1ua7h88kJ6dECih6uumLeOIV9a3QNdP-g&confirm=t&uuid=830d2994-960f-421f-be31-dc3a84b300c3
To: /content/drive/MyDrive/SetStop_KGQA/data/raw/nsm_cwq/CWQ.tgz.partial
100%|██████████| 914M/914M [00:09<00:00, 93.5MB/s]



Extracting the five files needed for this audit.

Recording source-file SHA-256 hashes.
These are provenance hashes, not a published checksum.
Hashing: train_simple.json
Hashing: dev_simple.json
Hashing: test_simple.json
Hashing: entities.txt
Hashing: relations.txt

Entity vocabulary: 2429346
Relation vocabulary: 6649

Auditing train...
train: 1,000 audited | 162 eligible | 2.1 minutes
train: 2,000 audited | 339 eligible | 2.3 minutes
train: 3,000 audited | 529 eligible | 2.6 minutes
train: 4,000 audited | 683 eligible | 2.9 minutes
train: 5,000 audited | 865 eligible | 3.2 minutes
train: 6,000 audited | 1,037 eligible | 3.5 minutes
train: 7,000 audited | 1,211 eligible | 3.8 minutes
train: 8,000 audited | 1,392 eligible | 4.2 minutes
train: 9,000 audited | 1,569 eligible | 4.5 minutes
train: 10,000 audited | 1,739 eligible | 4.9 minutes
train: 11,000 audited | 1,906 eligible | 5.2 minutes
train: 12,000 audited | 2,091 eligible | 5.6 minutes
train: 13,000 audited | 2,281 eligible | 6.